In [ ]:
"""Generic human-in-the-loop resume loop for DeepAgents interrupts.

Any agent using an HITL middleware (e.g. `middlewares.hitl_for(...)`) pauses
via a LangGraph interrupt instead of running unattended. This drives that
interrupt/resume cycle over the CLI so callers don't each reimplement it.
"""

import json
from collections.abc import Callable
from typing import Any

from langgraph.types import Command


def _get_messages_count(agent: Any, config: dict | None) -> int:
    if config is None:
        return 0
    try:
        state = agent.get_state(config)
        return len(state.values.get("messages", []))
    except Exception:  # noqa: BLE001
        return 0


def _print_new_tool_outputs(agent: Any, config: dict | None, prev_count: int) -> int:
    if config is None:
        return prev_count
    try:
        state = agent.get_state(config)
        messages = state.values.get("messages", [])
        for m in messages[prev_count:]:
            m_type = getattr(m, "type", "") or type(m).__name__.lower()
            if "tool" in m_type:
                name = getattr(m, "name", "tool")
                content = str(getattr(m, "content", "")).strip()
                if content:
                    print(f"\n[Tool Execution Output - {name}]:\n{content}\n")
        return len(messages)
    except Exception:  # noqa: BLE001
        return prev_count


def _format_prompt(
    assistant_prompt: str | Callable[[dict[str, Any]], str] | None,
    request: dict[str, Any],
) -> str:
    action_name = request.get("name", "action")
    if callable(assistant_prompt):
        return assistant_prompt(request)
    if isinstance(assistant_prompt, str):
        if "{name}" in assistant_prompt:
            return assistant_prompt.format(name=action_name)
        return assistant_prompt
    return f"\nApprove {action_name} action? [y/n]: "


def _handle_interrupt_decisions(
    interrupt: Any,
    assistant_prompt: str | Callable[[dict[str, Any]], str] | None = None,
) -> dict[str, Any]:
    decisions: list[dict[str, Any]] = []
    print("\n[Notice] Agent requested action requiring approval:")
    for request in interrupt.value.get("action_requests", []):
        action_name = request.get("name", "action")
        if request.get("description"):
            print(f"\n{request['description']}")
        else:
            print(f"\n[Action Review Request]: {action_name}")
            args = request.get("args", {})
            try:
                print(f"Arguments: {json.dumps(args, indent=2, default=str)}")
            except Exception:  # noqa: BLE001
                print(f"Arguments: {args}")

        prompt_text = _format_prompt(assistant_prompt, request)
        decision = input(prompt_text).strip().lower()
        if decision in ("y", "yes"):
            print(f"[Approved] Executing {action_name}...")
            decisions.append({"type": "approve"})
        else:
            reason = input("Reason for rejection (optional, press Enter to skip): ").strip()
            print(f"[Rejected] {action_name} rejected. Returning feedback to agent...")
            reject_payload: dict[str, Any] = {"type": "reject"}
            if reason:
                reject_payload["message"] = reason
            decisions.append(reject_payload)

    return {"decisions": decisions}


def run_with_hitl(
    agent: Any,
    initial_input: Any,
    config: dict | None = None,
    *,
    assistant_prompt: str | Callable[[dict[str, Any]], str] | None = None,
) -> dict[str, Any]:
    """Invoke `agent`, resolving any HITL interrupts via CLI y/n prompts.

    Loops until the agent produces a result with no pending interrupt.
    When an action is rejected, prompts for an optional reason and forwards
    it to the agent as actionable feedback for self-correction.
    """
    seen_msg_count = _get_messages_count(agent, config)

    if initial_input is None:
        if config is None:
            msg = "Config must be provided when resuming from existing state (initial_input=None)"
            raise ValueError(msg)
        state = agent.get_state(config)
        pending = [i for t in state.tasks for i in t.interrupts] if state.tasks else []
        if not pending:
            return state.values
        interrupt = pending[0]
        resume = _handle_interrupt_decisions(interrupt, assistant_prompt)
        result = agent.invoke(Command(resume=resume), config)
    else:
        result = agent.invoke(initial_input, config)

    seen_msg_count = _print_new_tool_outputs(agent, config, seen_msg_count)

    while "__interrupt__" in result:
        interrupt = result["__interrupt__"][0]
        resume = _handle_interrupt_decisions(interrupt, assistant_prompt)
        result = agent.invoke(Command(resume=resume), config)
        seen_msg_count = _print_new_tool_outputs(agent, config, seen_msg_count)

    return result



In [1]:
"""Universal session resolution & thread management utility for Agentic AI applications.

Supports 6 operational session modes:
1. 'default' (or empty/None): Uses the application's default thread ID.
2. 'new': Generates a unique, prefixed thread ID (e.g. `interpreter_a1b2c3`).
3. 'secret': Ephemeral in-memory execution bypassing persistence (`InMemorySaver`), leaving zero DB footprints.
4. 'resume': Automatically resumes the latest recorded checkpoint thread from storage.
5. 'step[N]': Resumes a recorded checkpoint thread by reverse-chronological 0-based index (`step[0]` = latest).
6. 'custom' (any other string): Directly adopts the user-provided thread ID name.
7. 'list' / 'history': Inspects and displays recent recorded sessions.
"""

from __future__ import annotations

import re
import sys
import uuid
from collections.abc import Callable
from dataclasses import dataclass
from typing import Any

from data.sqlite_db_data import get_recent_threads
from langgraph.checkpoint.memory import InMemorySaver


@dataclass(frozen=True)
class ResolvedSession:
    """Encapsulates the resolved runtime session configuration."""

    agent: Any
    thread_id: str
    mode_name: str
    is_secret: bool

    @property
    def config(self) -> dict[str, Any]:
        """Convenience dictionary for LangGraph / DeepAgents invocation."""
        return {"configurable": {"thread_id": self.thread_id}}


def format_session_list(
    threads: list[str],
    *,
    cli_command_hint: str = "python app.py step[<index>]",
) -> str:
    """Format a list of recorded session thread IDs into a clean readable banner."""
    if not threads:
        return "No recorded sessions found in database."

    lines = [
        "=" * 60,
        "Recorded Sessions (0-indexed, most recent first):",
    ]
    for i, tid in enumerate(threads):
        tag = " (latest)" if i == 0 else ""
        lines.append(f"  step[{i}]: {tid}{tag}")
    lines.append("=" * 60)
    lines.append(f"To resume: {cli_command_hint}")
    return "\n".join(lines)


def resolve_session(
    raw_arg: str | None,
    *,
    default_thread_id: str,
    default_agent: Any | None = None,
    agent_factory: Callable[[Any], Any] | None = None,
    app_name: str = "agent",
    get_threads_fn: Callable[..., list[str]] = get_recent_threads,
    cli_command_hint: str | None = None,
    handle_cli_exit: bool = True,
) -> ResolvedSession:
    """Resolve an incoming CLI argument or mode string into a fully configured `ResolvedSession`.

    Args:
        raw_arg: The argument string passed by caller (e.g. sys.argv[1]), mode name, or None.
        default_thread_id: The fallback/default thread ID for standard session.
        default_agent: Pre-initialized default agent instance (using persistent checkpointer).
        agent_factory: Optional callable `factory(checkpointer) -> agent` used to construct
            agents on the fly, essential for ephemeral `secret` mode with `InMemorySaver()`.
            If omitted and `secret` mode is requested, `default_agent` is used.
        app_name: Prefix used when generating unique thread IDs in `new` mode.
        get_threads_fn: Callable returning recorded thread IDs ordered descending by latest activity.
        cli_command_hint: Hint displayed in `list` mode and error messages. Defaults to
            `python core/apps/{app_name}_app.py step[<index>]`.
        handle_cli_exit: If True, exits via `sys.exit(0)` on `list` or `sys.exit(1)` on invalid
            step index. If False, raises ValueError or returns special sessions.

    Returns:
        A `ResolvedSession` instance containing (agent, thread_id, mode_name, is_secret).
    """
    arg = (raw_arg or "").strip()
    hint = cli_command_hint or f"python core/apps/{app_name}_app.py step[<index>]"

    def _resolve_agent(checkpointer: Any | None) -> Any:
        if agent_factory is not None:
            return agent_factory(checkpointer)
        return default_agent

    # 1. Default Mode
    if not arg or arg.lower() == "default":
        return ResolvedSession(
            agent=_resolve_agent(None),
            thread_id=default_thread_id,
            mode_name="default",
            is_secret=False,
        )

    # 2. New Unique Mode
    if arg.lower() == "new":
        tid = f"{app_name}_{uuid.uuid4().hex[:6]}"
        return ResolvedSession(
            agent=_resolve_agent(None),
            thread_id=tid,
            mode_name="new",
            is_secret=False,
        )

    # 3. Secret (Ephemeral In-Memory) Mode
    if arg.lower() == "secret":
        tid = f"secret_{uuid.uuid4().hex[:6]}"
        mem_saver = InMemorySaver()
        secret_agent = _resolve_agent(mem_saver)
        return ResolvedSession(
            agent=secret_agent,
            thread_id=tid,
            mode_name="secret (in-memory)",
            is_secret=True,
        )

    # 4. List / History Mode
    if arg.lower() in {"list", "history"}:
        threads = get_threads_fn(limit=15)
        output = format_session_list(threads, cli_command_hint=hint)
        print(output)
        if handle_cli_exit:
            sys.exit(0)
        return ResolvedSession(
            agent=_resolve_agent(None),
            thread_id=default_thread_id,
            mode_name="list",
            is_secret=False,
        )

    # 5. Resume (Latest Recorded Thread) Mode
    if arg.lower() == "resume":
        threads = get_threads_fn()
        if threads:
            return ResolvedSession(
                agent=_resolve_agent(None),
                thread_id=threads[0],
                mode_name=f"resume ({threads[0]})",
                is_secret=False,
            )
        print("[Notice] No previous sessions found in database. Using default.")
        return ResolvedSession(
            agent=_resolve_agent(None),
            thread_id=default_thread_id,
            mode_name="default (no prior sessions)",
            is_secret=False,
        )

    # 6. Step[Index] Mode (Reverse-chronological 0-based index)
    step_match = re.match(r"^step\[?(\d+)\]?$", arg, re.IGNORECASE)
    if step_match:
        idx = int(step_match.group(1))
        threads = get_threads_fn()
        if not threads:
            msg = "[Error] No recorded sessions found in database."
            print(msg)
            if handle_cli_exit:
                sys.exit(1)
            raise ValueError(msg)

        if 0 <= idx < len(threads):
            selected = threads[idx]
            return ResolvedSession(
                agent=_resolve_agent(None),
                thread_id=selected,
                mode_name=f"step[{idx}] ({selected})",
                is_secret=False,
            )

        err_msg = (
            f"[Error] Index {idx} out of range. Available: step[0] to step[{len(threads) - 1}].\n"
            "Recent sessions:\n"
            + "\n".join(f"  step[{i}]: {t}" for i, t in enumerate(threads[:5]))
        )
        print(err_msg)
        if handle_cli_exit:
            sys.exit(1)
        raise IndexError(err_msg)

    # 7. Custom Named Session
    return ResolvedSession(
        agent=_resolve_agent(None),
        thread_id=arg,
        mode_name=f"custom ({arg})",
        is_secret=False,
    )


In [2]:
from data.sqlite_db_data import get_sqlite_checkpointer
from graphs.code_review_parentgraph import build_code_review_graph

session = resolve_session(
    raw_arg="test",
    default_thread_id="cli_review_session",
    agent_factory=lambda saver: build_code_review_graph(checkpointer=saver or get_sqlite_checkpointer()),
    app_name="code_review",
    cli_command_hint="python core/apps/code_review_app.py [file] step[<index>]",
)

In [3]:
print(session)

ResolvedSession(agent=<langgraph.graph.state.CompiledStateGraph object at 0x000001D6BE665A90>, thread_id='test', mode_name='custom (test)', is_secret=False)
